# Heart Disease Prediction - My Submission
Testing out some models to see if we can predict heart disease.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Task 1: Loading
data = pd.read_csv('q1_heart_disease.csv')
print(data.shape)
print(data.dtypes)
print(data.isnull().sum())
data.head()

## EDA stuff
Just looking at the data.

In [ ]:
# Target distribution
sns.countplot(x='heart_disease', data=data)
plt.title('Count of Disease vs No Disease')
plt.show()

# Heatmap
plt.figure(figsize=(10,8))
sns.heatmap(data.corr(), annot=True)
plt.show()

# Age vs Heart Disease
sns.boxplot(x='heart_disease', y='age', data=data)
plt.show()

**Interpretations:**
1. The target is mostly balanced, maybe a bit more 0s than 1s.
2. Max heart rate (max_hr) seems to have a strong negative correlation with disease.
3. Older people seem slightly more likely to have it, but there's a lot of overlap.

In [ ]:
# Task 3: Preprocessing
# dropping rows with nulls because it's easier
data = data.dropna()

# One hot encoding
# I'll just list the ones that look like categories
cats = ['chest_pain_type', 'resting_ecg', 'st_slope']
data_final = pd.get_dummies(data, columns=cats)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data_final.drop('heart_disease', axis=1)
y = data_final['heart_disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Dropped missing rows to keep it simple. Used one-hot encoding on the text columns and scaled everything.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
pred1 = dt.predict(X_test_scaled)
print("--- Decision Tree ---")
print(confusion_matrix(y_test, pred1))
print(classification_report(y_test, pred1))

# RF
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
pred2 = rf.predict(X_test_scaled)
print("--- Random Forest ---")
print(confusion_matrix(y_test, pred2))
print(classification_report(y_test, pred2))

# GB
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_scaled, y_train)
pred3 = gb.predict(X_test_scaled)
print("--- Gradient Boosting ---")
print(confusion_matrix(y_test, pred3))
print(classification_report(y_test, pred3))

**Best Model:**
Random Forest seems to be the winner here. It has a higher F1-score than the others, and the recall for class 1 is better, which is important because we don't want to miss sick people.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Task 6: Tuning RF
params = {'n_estimators': [50, 100, 200]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), params, cv=3)
grid.fit(X_train_scaled, y_train)

print("Best params:", grid.best_params_)
new_pred = grid.predict(X_test_scaled)
print("Tuned report:")
print(classification_report(y_test, new_pred))